# 🚀 Google Colab CPU Benchmark & Pipeline Smoke Test (Milestone 1)

Notebook này dùng để chạy thử nghiệm môi trường và đo benchmark tài nguyên chính thức trên Google Colab CPU (Linux, 2 vCPU, 12GB RAM) theo chuẩn Schema 1.2.

> **Hướng dẫn:** Chọn runtime **CPU** (Runtime > Change runtime type > CPU), sau đó thực thi tuần tự các cell bên dưới.

## 1. Thiết lập Môi trường & Dependencies

In [ ]:
# Cài đặt các thư viện cần thiết (psutil dùng để đo RAM cây tiến trình)
!pip install psutil matplotlib

import os
import sys

# Thêm thư mục src vào PYTHONPATH
sys.path.insert(0, os.path.abspath('src'))
os.environ['PYTHONPATH'] = os.path.abspath('src')
print('Môi trường sẵn sàng. Python:', sys.version)

## 2. Chạy Kiểm thử Đơn vị T6 (Statistics & Benchmark)

In [ ]:
!python -m unittest tests/test_statistics.py tests/test_benchmark.py -v

## 3. Thực thi Pipeline kèm Đo lường Tài nguyên (Benchmark)

In [ ]:
!mkdir -p /content/data/colab-baseline
!mkdir -p /content/benchmarks
!mkdir -p /content/statistics

# Chạy benchmark với lệnh runner (5 bài trong dải Trọng)
!python -m arxiv_pipeline.benchmark \
  --cmd "python -m arxiv_pipeline.runner --owner trong --run-id colab-baseline --max-candidates 10 --max-papers 5 --output-root /content/data/colab-baseline" \
  --run-id colab-baseline \
  --output-root /content/data/colab-baseline \
  --output-dir /content/benchmarks \
  --interval 0.1

## 4. Tạo Thống kê Schema 1.2 (Statistics)

In [ ]:
!python -m arxiv_pipeline.statistics \
  --output-root /content/data/colab-baseline \
  --run-id colab-baseline \
  --output-dir /content/statistics

# Hiển thị kết quả JSON thống kê
!cat /content/statistics/colab-baseline.json

## 5. Đọc & Hiển thị Kết quả Benchmark & Timing Sidecar

In [ ]:
import json
from pathlib import Path

bench_path = Path('/content/benchmarks/colab-baseline.json')
timing_path = Path('/content/benchmarks/colab-baseline-timing.json')

if bench_path.exists():
    print('=== BENCHMARK REPORT (Schema 1.2) ===')
    bench = json.loads(bench_path.read_text())
    print(json.dumps(bench, indent=2))

if timing_path.exists():
    print('\n=== TIMING SIDECAR (Invocation Measurements) ===')
    timings = json.loads(timing_path.read_text())
    print(json.dumps(timings, indent=2))

## 6. Đóng gói Artifacts Kết quả (.tar.gz)

In [ ]:
!tar -czvf colab_baseline_results.tar.gz -C /content data/colab-baseline benchmarks statistics
!sha256sum colab_baseline_results.tar.gz
print('Gói kết quả đã sẵn sàng để tải về máy.')